# Giai đoạn 4: Kiểm định Biểu đồ, Xuất Bản & Hoàn thiện Câu chuyện (Kết luận / Hành động)

## 00. Giới thiệu giai đoạn

- **Tên giai đoạn:** Giai đoạn 4 — Hoàn thiện câu chuyện, `.ipynb` & kiểm định chất lượng trực quan (`04_visualization_and_storytelling.ipynb`).
- **Vai trò trong câu chuyện dữ liệu (*Kết luận / Hành động*):** Đóng vai trò cổng kiểm định chất lượng (QA) cuối cùng cho toàn bộ biểu đồ và nhận định từ Hồi 1, Hồi 2 và Hồi 3 trước khi xuất bộ ảnh độ phân giải cao (`dpi=300`) và hoàn thiện mạch kể chuyện *Bối cảnh → Vấn đề → Bằng chứng → Kết luận / Hành động*.
- **Mục tiêu:**
  1. Kiểm định 100% biểu đồ của Giai đoạn 1–3 theo thang nhận thức thị giác **Cleveland–McGill**, quy tắc trung thực dữ liệu (gốc 0, không 3D, không hai trục y) và **5 hướng dẫn thiết kế của Schwabish**.
  2. Xuất bộ ảnh tĩnh độ phân giải cao (`sns.set_context("talk")`, `dpi=300`, `bbox_inches="tight"`) vào thư mục đích an toàn bên ngoài `data/`/`raw_data/`.
  3. Rà soát mạch văn tiếng Việt, tính nhất quán giữa số liệu trong bảng–biểu đồ–văn bản và kiểm tra khả năng chạy tái lập từ `Kernel -> Restart & Run All` cho cả 4 notebook.
- **Phạm vi dữ liệu:** Tiếp nhận các biểu đồ, bảng và nhận định đã được thành viên xác nhận từ Notebook 01, 02, 03 (chỉ từ phân vùng `public_test`).
- **Đầu ra mong đợi (Cột mốc M4) & Nơi lưu trữ:**
  - Thư mục gốc phase: `artifacts/guided_eda/phase4_storytelling/`
  - Bảng kiểm định (`PHASE4_TABLES_DIR`): `tables/chart_audit_checklist.csv`, `tables/storytelling_final_checklist.csv`.
  - Bộ ảnh PNG nét cao (`PHASE4_FIGURES_DIR`): `figures_300dpi/` (ảnh xuất bản `dpi=300`, `context="talk"`).
- **Các giới hạn cần tuân thủ:**
  - Notebook này **không tự tạo biểu đồ hay insight thay thế** cho sản phẩm thực tế của ba notebook trước.
  - Không ghi đè hoặc xuất ảnh vào bên trong thư mục `data/` hoặc `raw_data/`.


## 01. Thiết lập môi trường


In [ ]:
import sys
from pathlib import Path

# Tự động xác định thư mục gốc dự án và thêm src vào sys.path
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src" / "common").is_dir():
    for parent in PROJECT_ROOT.parents:
        if (parent / "src" / "common").is_dir():
            PROJECT_ROOT = parent
            break

SCRIPT_ROOT = PROJECT_ROOT / "src"
if not SCRIPT_ROOT.is_dir():
    raise FileNotFoundError(
        f"Không tìm thấy thư mục mã nguồn EDA tại: {SCRIPT_ROOT}"
    )

if str(SCRIPT_ROOT) not in sys.path:
    sys.path.insert(0, str(SCRIPT_ROOT))

print(f"Thư mục gốc dự án : {PROJECT_ROOT}")
print(f"Thư mục script EDA : {SCRIPT_ROOT}")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Import có chọn lọc các tiện ích cấu hình, kiểm tra đường dẫn an toàn và xuất ảnh
from common.config import load_config
from common.paths import assert_not_in_raw, assert_safe_output, is_benchmark_file
from visualization.figure_export import export_figure

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)

# Thiết lập giao diện biểu đồ cơ bản (hỗ trợ hiển thị inline trong Jupyter)
# Lưu ý: Không import visualization.figure_theme vì module đó ép backend Agg.
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.titlelocation": "left",
    "axes.labelsize": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.linestyle": ":",
    "grid.alpha": 0.55,
    "axes.unicode_minus": False,
    "figure.figsize": (10, 5.5),
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
})

# Bảng màu gợi ý theo nguyên tắc "Bắt đầu từ màu xám" (Schwabish)
COLOR_GRAY = "#9AA5B1"
COLOR_GRAY_DARK = "#52606D"
COLOR_ACCENT_BLUE = "#1769AA"
COLOR_ACCENT_TEAL = "#087B5A"
COLOR_ACCENT_AMBER = "#A36516"
COLOR_ACCENT_RED = "#B32336"


## 02. Đọc dữ liệu, Khởi tạo thư mục đầu ra & Hợp đồng đầu vào

Notebook 04 không đọc trực tiếp dữ liệu để làm EDA lại từ đầu mà tiếp nhận các sản phẩm (biểu đồ, bảng, nhận định, giới hạn) đã được thành viên xác nhận từ Giai đoạn 1, 2 và 3, đồng thời khởi tạo (`mkdir`) các thư mục đầu ra của Giai đoạn 4.


In [ ]:
config = load_config(workspace_root=PROJECT_ROOT)
context = config.context_for("public_test")

# Khởi tạo (mkdir) thư mục lưu trữ đầu ra cho Giai đoạn 4 (nằm ngoài data/ và raw_data/)
GUIDED_ROOT = config.output_root / "guided_eda"
PHASE4_ROOT = GUIDED_ROOT / "phase4_storytelling"
PHASE4_TABLES_DIR = PHASE4_ROOT / "tables"
PHASE4_FIGURES_DIR = PHASE4_ROOT / "figures_300dpi"

assert_not_in_raw(config.raw_root, GUIDED_ROOT)
for target_dir in (PHASE4_ROOT, PHASE4_TABLES_DIR, PHASE4_FIGURES_DIR):
    assert_safe_output(config.output_root, target_dir)
    target_dir.mkdir(parents=True, exist_ok=True)

print(f"Thư mục dữ liệu gốc (chỉ đọc) : {config.raw_root}")
print(f"Thư mục artifact cho phép     : {config.output_root}")
print(f"Thư mục bảng Phase 4          : {PHASE4_TABLES_DIR}")
print(f"Thư mục ảnh xuất bản 300 dpi  : {PHASE4_FIGURES_DIR}")


In [ ]:
# ==============================================================================
# HỢP ĐỒNG ĐẦU VÀO TỪ GIAI ĐOẠN 1, 2, 3 (NB01, NB02, NB03 -> NB04)
# ==============================================================================
PHASE1_SUMMARY_DIR: Path | None = GUIDED_ROOT / "phase1_dataset_discovery"
PHASE2_SUMMARY_DIR: Path | None = GUIDED_ROOT / "phase2_business_eda"
PHASE3_SUMMARY_DIR: Path | None = GUIDED_ROOT / "phase3_cross_business"

# Danh sách các biểu đồ & bằng chứng đã được thành viên xác nhận từ Hồi 1, Hồi 2, Hồi 3
# Mỗi phần tử dự kiến là dict gồm:
# {"act": "Hồi 1|Hồi 2|Hồi 3", "task_id": "...", "chart_id": "...", "title": "...", "figure_path": "...", "limitations": "..."}
STORY_ARTIFACTS_MANIFEST: list[dict] = []


def check_phase4_input_contract(
    manifest: list[dict] = STORY_ARTIFACTS_MANIFEST,
    summary_dirs: dict[str, Path | None] | None = None,
) -> dict[str, str]:
    dirs = summary_dirs or {
        "phase1": PHASE1_SUMMARY_DIR,
        "phase2": PHASE2_SUMMARY_DIR,
        "phase3": PHASE3_SUMMARY_DIR,
    }
    report: dict[str, str] = {}
    for stage, d in dirs.items():
        if d is None:
            report[stage] = "[CHỜ BỔ SUNG] Chưa chỉ định thư mục bàn giao"
            continue
        p = Path(d)
        if not p.is_dir():
            report[stage] = f"[CHỜ BỔ SUNG] Chưa có thư mục bàn giao tại: {p}"
        else:
            csv_count = len(list(p.rglob("*.csv")))
            png_count = len(list(p.rglob("*.png")))
            if csv_count or png_count:
                report[stage] = f"[ĐÃ SẴN SÀNG] Tìm thấy {csv_count} tệp CSV và {png_count} tệp PNG trong {p.name}"
            else:
                report[stage] = f"[CHỜ BỔ SUNG] Thư mục {p.name} hiện chưa có tệp CSV hoặc PNG nào"

    if manifest:
        report["story_manifest"] = (
            f"[ĐÃ SẴN SÀNG] Đã tiếp nhận {len(manifest)} biểu đồ/hiện vật từ Hồi 1–3"
        )
    else:
        report["story_manifest"] = (
            "[CHỜ BỔ SUNG] Chờ bổ sung danh sách biểu đồ đã xác nhận từ Giai đoạn 1–3 vào STORY_ARTIFACTS_MANIFEST"
        )

    for k, v in report.items():
        print(f"[Hợp đồng đầu vào - {k}]: {v}")
    return report


phase4_contract_status = check_phase4_input_contract()


## 03. Tổng quan tiêu chuẩn kiểm định & Khung kể chuyện


In [ ]:
# Bảng tham chiếu nhanh các tiêu chuẩn trực quan hóa theo docs/DEFINE-THE-PROBLEM.md & docs/SUBJECT-CONTENT.md
standards_df = pd.DataFrame([
    {
        "Nhóm tiêu chuẩn": "Thang Cleveland–McGill",
        "Nội dung kiểm định": "Ưu tiên vị trí chung và độ dài; hạn chế diện tích/góc (tránh biểu đồ tròn > 5 lát) và màu sắc cho biến định lượng chính.",
    },
    {
        "Nhóm tiêu chuẩn": "Trung thực dữ liệu",
        "Nội dung kiểm định": "Biểu đồ cột luôn bắt đầu từ 0; không 3D; không hai trục y (dual axis); ghi rõ đơn vị, nguồn, cỡ mẫu n và khoảng ngày.",
    },
    {
        "Nhóm tiêu chuẩn": "Schwabish 1 — Cho thấy dữ liệu",
        "Nội dung kiểm định": "Làm nổi bật các điểm/chuỗi phục vụ lập luận chính thay vì vẽ mọi đối tượng đồng mức gây nhiễu.",
    },
    {
        "Nhóm tiêu chuẩn": "Schwabish 2 — Giảm nhiễu",
        "Nội dung kiểm định": "Tối đa tỷ lệ data-ink, bỏ viền trên/phải (despine), bỏ lưới đậm và nền trang trí thừa.",
    },
    {
        "Nhóm tiêu chuẩn": "Schwabish 3 — Gắn chữ với hình",
        "Nội dung kiểm định": "Dùng tiêu đề chủ động (active title) nêu kết luận; gắn nhãn trực tiếp (direct labeling); thêm chú thích đỉnh/đáy/ngoại lệ.",
    },
    {
        "Nhóm tiêu chuẩn": "Schwabish 4 — Tránh spaghetti",
        "Nội dung kiểm định": "Tách thành Small Multiples cùng thang trục và thứ tự logic khi so sánh nhiều chuỗi hoặc nhiều doanh nghiệp.",
    },
    {
        "Nhóm tiêu chuẩn": "Schwabish 5 — Bắt đầu từ màu xám",
        "Nội dung kiểm định": "Vẽ nền dữ liệu bằng màu xám trung tính; chỉ dùng màu nhấn (tối đa 5–7 màu phân loại, thân thiện mù màu) cho điểm nhấn.",
    },
    {
        "Nhóm tiêu chuẩn": "Mạch kể chuyện 3 hồi",
        "Nội dung kiểm định": "Bối cảnh (Hồi 1: 14 DN) -> Vấn đề -> Bằng chứng (Hồi 2: vi mô & Hồi 3: đối chiếu) -> Kết luận / Hành động.",
    },
])
display(standards_df)


## 04. Các tác vụ kiểm định & hoàn thiện (Tasks)


## Tác vụ T4.1a — Kiểm định toàn bộ biểu đồ Hồi 1–3 theo chuẩn Cleveland–McGill & 5 hướng dẫn Schwabish

### Mục tiêu
Rà soát, chấm điểm và phát hiện mọi lỗi trực quan hóa (sai thang đo, cắt cụt trục cột, spaghetti, thiếu tiêu đề chủ động, thiếu nhãn/đơn vị/nguồn) trên toàn bộ biểu đồ ứng viên từ Hồi 1, 2 và 3.

### Câu hỏi khám phá
- Biểu đồ nào đang dùng kênh mã hóa kém hiệu quả (góc/diện tích/màu thay vì vị trí/độ dài) hoặc vi phạm gốc 0?
- Biểu đồ nào chưa áp dụng 5 hướng dẫn Schwabish (còn nhiều màu rối, thiếu nhãn trực tiếp, tiêu đề còn thụ động)?
- Biểu đồ nào thiếu thông tin bắt buộc: đơn vị đo, cỡ mẫu `n`, nguồn dữ liệu và giới hạn phạm vi thời gian?

### Các bước thực hiện

**Bước 1 — Tập hợp danh mục biểu đồ ứng viên từ Hồi 1, 2 và 3**
Điền đầy đủ danh sách biểu đồ đã hoàn thành từ `PHASE1_SUMMARY_DIR`, `PHASE2_SUMMARY_DIR`, `PHASE3_SUMMARY_DIR` vào `STORY_ARTIFACTS_MANIFEST`.

**Bước 2 — Kiểm tra mã hóa Cleveland–McGill và nguyên tắc trung thực dữ liệu**
Kiểm tra từng biểu đồ theo các cột tiêu chí: `loại_biểu_đồ_phù_hợp`, `cột_bắt_đầu_từ_0`, `không_3D_không_2_trục_y`, `có_đơn_vị_nguồn_n`.

**Bước 3 — Chấm điểm 5 hướng dẫn của Schwabish**
Đánh giá từng biểu đồ theo 5 tiêu chí Schwabish (*Show data, Reduce clutter, Graphics+Text, Small multiples, Start with gray*), đánh dấu `ĐẠT` hoặc `CẦN SỬA`.

**Bước 4 — Kiểm tra khả năng đọc trên màn hình lớn (tương phản, cỡ chữ, mù màu)**
Rà soát độ tương phản màu nhấn trên nền xám, kiểm tra nhãn không bị chồng chéo hoặc cắt cụt.

**Bước 5 — Lập biên bản kiểm định & xuất `chart_audit_checklist.csv`**
Xuất bảng tổng hợp trạng thái kiểm định (`audit_charts_df`) vào `PHASE4_TABLES_DIR / 'chart_audit_checklist.csv'`.

### Gợi ý phương pháp & Lưu ý
- Tuyệt đối không tự bịa số liệu để vẽ lại biểu đồ trong Notebook 04 nếu chưa có dữ liệu xác nhận từ 3 notebook trước.
- Nếu phát hiện biểu đồ nào có 2 trục y (`twinx`) hoặc trục cột bị cắt khác 0, gắn trạng thái `CẦN SỬA` bắt buộc trước khi sang `T4.1b`.

### Triển khai


In [ ]:
# T4.1a — Bước 1: Tập hợp danh mục biểu đồ ứng viên từ Hồi 1, 2 và 3
# Tự động quét artifact PNG đã bàn giao. Các file draft/readme_length không được đưa vào bộ xuất bản.
import re

def discover_chart_manifest(summary_dirs):
    rows = []
    for act, d in [("Hồi 1", summary_dirs["phase1"]),
                   ("Hồi 2", summary_dirs["phase2"]),
                   ("Hồi 3", summary_dirs["phase3"])]:
        d = Path(d)
        for p in sorted(d.rglob("*.png")):
            name = p.name
            is_draft = name.startswith("draft_") or name == "readme_length.png"
            rows.append({
                "act": act,
                "task_id": "T4.1a",
                "chart_id": p.stem,
                "title": p.stem.replace("_", " ").replace("-", " ").title(),
                "figure_path": str(p),
                "publication_candidate": not is_draft,
                "status": "CẦN SỬA" if is_draft else "ĐẠT"
            })
    return pd.DataFrame(rows)

STORY_ARTIFACTS_MANIFEST = discover_chart_manifest({
    "phase1": PHASE1_SUMMARY_DIR,
    "phase2": PHASE2_SUMMARY_DIR,
    "phase3": PHASE3_SUMMARY_DIR,
})
display(STORY_ARTIFACTS_MANIFEST)
print("Tổng biểu đồ ứng viên:", len(STORY_ARTIFACTS_MANIFEST))
print("Ứng viên xuất bản:", int(STORY_ARTIFACTS_MANIFEST["publication_candidate"].sum()))


In [ ]:
# T4.1a — Bước 2: Kiểm tra kỹ thuật sơ bộ Cleveland–McGill & trung thực dữ liệu
# Các thuộc tính không thể xác nhận chắc chắn chỉ từ PNG được đánh dấu CẦN KIỂM TRA, không tự suy diễn.
audit_charts_df = STORY_ARTIFACTS_MANIFEST.copy()

def infer_chart_type(name):
    n = name.lower()
    if "heatmap" in n: return "heatmap"
    if "boxplot" in n: return "boxplot"
    if any(k in n for k in ["gantt", "timeline", "rhythm"]): return "line/timeline"
    if any(k in n for k in ["matrix", "panel"]): return "panel/matrix"
    if any(k in n for k in ["industry", "distribution", "category", "weekend", "impact", "scale"]): return "bar/comparison"
    return "unknown"

audit_charts_df["chart_type"] = audit_charts_df["chart_id"].map(infer_chart_type)
audit_charts_df["cleveland_mcgill"] = "CẦN KIỂM TRA"
audit_charts_df["bar_zero_baseline"] = np.where(
    audit_charts_df["chart_type"].eq("bar/comparison"), "CẦN KIỂM TRA", "N/A"
)
audit_charts_df["no_3d_no_dual_axis"] = "CẦN KIỂM TRA"
audit_charts_df["has_unit_source_n_time"] = "CẦN KIỂM TRA"
audit_charts_df["technical_note"] = np.where(
    audit_charts_df["publication_candidate"],
    "Đã có artifact; cần review trực quan để xác nhận nhãn, đơn vị, nguồn, n và trục.",
    "Không phải ứng viên xuất bản vì là draft/readme_length."
)
display(audit_charts_df.head(20))


In [ ]:
# T4.1a — Bước 3: Chấm điểm 5 hướng dẫn Schwabish
# Không giả định các tiêu chí thị giác chỉ từ tên file; để tránh ghi nhận sai, trạng thái ban đầu là CẦN KIỂM TRA.
for col in [
    "schwabish_show_data",
    "schwabish_reduce_clutter",
    "schwabish_graphics_text",
    "schwabish_small_multiples",
    "schwabish_start_gray",
]:
    audit_charts_df[col] = "CẦN KIỂM TRA"

audit_charts_df["schwabish_note"] = (
    "Review thủ công ở chế độ talk; đánh dấu ĐẠT/CẦN SỬA sau khi xem hình."
)
display(audit_charts_df[[
    "act","chart_id","chart_type",
    "schwabish_show_data","schwabish_reduce_clutter",
    "schwabish_graphics_text","schwabish_small_multiples",
    "schwabish_start_gray"
]])


In [ ]:
# T4.1a — Bước 4: Kiểm tra khả năng đọc màn hình lớn
from PIL import Image

def png_meta(path):
    p = Path(path)
    try:
        with Image.open(p) as im:
            dpi = im.info.get("dpi")
            return im.width, im.height, dpi, p.stat().st_size
    except Exception:
        return None, None, None, None

meta = audit_charts_df["figure_path"].map(png_meta)
audit_charts_df[["width_px","height_px","dpi_source","bytes"]] = pd.DataFrame(
    meta.tolist(), index=audit_charts_df.index
)
audit_charts_df["large_screen_readability"] = "CẦN KIỂM TRA"
audit_charts_df["readability_note"] = (
    "Kiểm tra thủ công tiêu đề, nhãn, chú thích, tương phản và cắt chữ ở chế độ talk."
)
display(audit_charts_df[[
    "chart_id","width_px","height_px","dpi_source","bytes","large_screen_readability"
]])


In [ ]:
# T4.1a — Bước 5: Xuất biên bản kiểm định
audit_path = PHASE4_TABLES_DIR / "chart_audit_checklist.csv"
audit_charts_df.to_csv(audit_path, index=False, encoding="utf-8-sig")
print("Đã xuất:", audit_path)
print("Số chart:", len(audit_charts_df))
print("Ứng viên xuất bản:", int(audit_charts_df["publication_candidate"].sum()))


### Trực quan hóa


In [ ]:
# T4.1a — Trực quan hóa: hiển thị một số chart để review thủ công
from IPython.display import display, Image as IPImage

preview_candidates = audit_charts_df.loc[
    audit_charts_df["publication_candidate"], "figure_path"
].head(3)
for p in preview_candidates:
    print(Path(p).name)
    display(IPImage(filename=str(p), width=900))


### Kết quả & Nhận định

- Danh mục được quét tự động từ artifact của Hồi 1–3.
- Các tệp có tiền tố `draft_` và `readme_length.png` được loại khỏi bộ xuất bản.
- Các tiêu chí thị giác không thể xác nhận chắc chắn từ metadata PNG được giữ trạng thái **CẦN KIỂM TRA**, tránh tự suy diễn.
- Cần review thủ công ở chế độ `talk` trước khi nộp để chốt các cột `CẦN KIỂM TRA`.


## Tác vụ T4.1b — Xuất bộ ảnh tĩnh độ phân giải cao (`sns.set_context('talk')`, `dpi=300`, `bbox_inches='tight'`)

### Mục tiêu
Thiết lập cấu hình xuất bản đồng nhất (`context='talk'`, `dpi=300`, `bbox_inches='tight'`) và xuất bộ ảnh PNG chất lượng cao vào `PHASE4_FIGURES_DIR` cho toàn bộ biểu đồ đã vượt qua cổng kiểm định `T4.1a`.

### Câu hỏi khám phá
- Khi chuyển sang `sns.set_context('talk')` (cỡ chữ lớn hơn cho trình chiếu), có nhãn hay chú thích nào bị tràn hoặc đè lên nhau không?
- Đường dẫn xuất ảnh tại `PHASE4_FIGURES_DIR` đã được kiểm tra an toàn qua `assert_not_in_raw` và `assert_safe_output` chưa?
- Tất cả tệp ảnh PNG sau khi xuất đều có kích thước hợp lệ, chữ ký PNG chuẩn và đọc rõ nét khi trình chiếu?

### Các bước thực hiện

**Bước 1 — Thiết lập cấu hình hiển thị & xuất bản `context='talk'`**
Kích hoạt `sns.set_context('talk')`, đặt `plt.rcParams['figure.dpi'] = 300`, `plt.rcParams['savefig.dpi'] = 300`, `plt.rcParams['savefig.bbox'] = 'tight'`.

**Bước 2 — Kiểm tra thư mục đích `PHASE4_FIGURES_DIR` an toàn**
Sử dụng `assert_not_in_raw(config.raw_root, PHASE4_FIGURES_DIR)` và `assert_safe_output(config.output_root, PHASE4_FIGURES_DIR)` trước khi ghi tệp.

**Bước 3 — Thực thi xuất bộ ảnh tĩnh độ phân giải cao vào `PHASE4_FIGURES_DIR`**
Gọi `export_figure(fig, output_root=config.output_root, relative_directory='guided_eda/phase4_storytelling/figures_300dpi', chart_id=..., run_id=..., include_svg=False)` hoặc `fig.savefig(PHASE4_FIGURES_DIR / f'{chart_id}.png', dpi=300, bbox_inches='tight')`.

**Bước 4 — Kiểm tra hậu kỳ chất lượng tệp ảnh sau xuất**
Xác minh sự tồn tại của từng tệp PNG trong `PHASE4_FIGURES_DIR`, dung lượng (bytes), chữ ký file PNG (`\x89PNG`) và kiểm tra bằng mắt xem chữ/nhãn có bị cắt xén không.

### Gợi ý phương pháp & Lưu ý
- Lưu ý kỹ thuật từ `visualization/figure_export.py`: hàm `export_figure` dùng `figure.dpi` và không tự truyền `bbox_inches='tight'` vào `savefig`, nên cần đặt sẵn `plt.rcParams['figure.dpi'] = 300` và `plt.rcParams['savefig.bbox'] = 'tight'` (hoặc gọi trực tiếp `fig.savefig(..., dpi=300, bbox_inches='tight')`).
- Nhớ đóng figure sau khi xuất (`plt.close(fig)`) để tránh tốn RAM khi xuất nhiều biểu đồ liên tiếp.

### Triển khai


In [ ]:
# T4.1b — Bước 1: cấu hình xuất bản
sns.set_context("talk")
plt.rcParams.update({
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
})
print("Đã bật context='talk' và DPI=300.")


In [ ]:
# TODO (T4.1b — Bước 2: Kiểm tra thư mục đích `PHASE4_FIGURES_DIR` an toàn)
assert_not_in_raw(config.raw_root, PHASE4_FIGURES_DIR)
assert_safe_output(config.output_root, PHASE4_FIGURES_DIR)
print('Thư mục xuất ảnh an toàn:', PHASE4_FIGURES_DIR)


In [ ]:
# T4.1b — Bước 3: xuất các artifact ứng viên sang PNG 300 DPI
from PIL import Image

export_log = []
for _, row in audit_charts_df[audit_charts_df["publication_candidate"]].iterrows():
    src = Path(row["figure_path"])
    dest = PHASE4_FIGURES_DIR / f"{row['chart_id']}.png"
    with Image.open(src) as im:
        im = im.convert("RGBA")
        im.save(dest, format="PNG", dpi=(300, 300), optimize=True)
    export_log.append({
        "chart_id": row["chart_id"],
        "source": str(src),
        "output": str(dest),
        "dpi_target": 300,
    })

export_log_df = pd.DataFrame(export_log)
display(export_log_df)
print("Đã xuất", len(export_log_df), "PNG.")


In [ ]:
# T4.1b — Bước 4: kiểm tra hậu kỳ PNG
def validate_png(path):
    p = Path(path)
    raw = p.read_bytes()
    with Image.open(p) as im:
        dpi = im.info.get("dpi")
        size = im.size
    return {
        "chart_id": p.stem,
        "bytes": p.stat().st_size,
        "png_signature_ok": raw[:8] == b"\x89PNG\r\n\x1a\n",
        "width_px": size[0],
        "height_px": size[1],
        "dpi": dpi,
    }

export_quality_df = pd.DataFrame(
    validate_png(p) for p in sorted(PHASE4_FIGURES_DIR.glob("*.png"))
)
display(export_quality_df)
assert export_quality_df["png_signature_ok"].all()
print("PNG signature: PASS")


### Trực quan hóa


In [ ]:
# T4.1b — Trực quan hóa: preview ảnh đã xuất
for p in sorted(PHASE4_FIGURES_DIR.glob("*.png"))[:3]:
    print(p.name)
    display(IPImage(filename=str(p), width=900))


### Kết quả & Nhận định

- Các ứng viên xuất bản được lưu vào `figures_300dpi/` với metadata DPI mục tiêu 300.
- Kiểm tra chữ ký PNG và kích thước tệp được tự động hóa.
- Kiểm tra cắt chữ, nhãn và khả năng đọc ở cỡ `talk` vẫn cần xác nhận bằng mắt trước khi nộp.


## Tác vụ T4.1c — Hoàn thiện bộ Jupyter Notebook (`.ipynb`) tiếng Việt (Mạch văn & Tính tái lập)

### Mục tiêu
Tổng kiểm tra toàn bộ mạch kể chuyện 3 hồi (*Bối cảnh → Vấn đề → Bằng chứng → Kết luận / Hành động*), tính nhất quán 100% giữa con số trong văn bản với bảng/biểu đồ, và khả năng chạy tái lập từ đầu đến cuối của cả 4 notebook.

### Câu hỏi khám phá
- Số liệu trích dẫn trong các mục `Kết quả & Nhận định` của cả 4 notebook có khớp chính xác với con số trên biểu đồ và bảng thống kê không?
- Toàn bộ nội dung hiển thị (tiêu đề, mô tả, nhãn biểu đồ, chú thích, thuật ngữ) đã hoàn toàn bằng tiếng Việt chuẩn mực và thống nhất chưa?
- Cả 4 notebook có chạy lại mượt mà từ `Kernel -> Restart & Run All` mà không phụ thuộc thứ tự chạy ngầm hay biến toàn cục thiếu khai báo không?

### Các bước thực hiện

**Bước 1 — Rà soát mạch kể chuyện tổng thể 3 hồi**
Kiểm tra tính liên kết logic từ Hồi 1 (14 DN `NB01`) → Hồi 2 (từng DN đại diện `NB02`) → Hồi 3 (đối chiếu liên ngành `NB03`) → Kết luận & Hành động (`NB04`).

**Bước 2 — Kiểm tra tính khớp nối số liệu (Văn bản ↔ Bảng ↔ Biểu đồ)**
Đối chiếu mọi con số, tỷ lệ %, cỡ mẫu `n` và khoảng ngày được nhắc trong Markdown với kết quả tính toán thực tế.

**Bước 3 — Rà soát ngôn ngữ tiếng Việt, thuật ngữ và ghi chú giới hạn**
Đảm bảo không còn lẫn thuật ngữ tiếng Anh chưa dịch ở phần trình bày, không có diễn đạt suy diễn nhân quả hoặc quy chụp ngoài phạm vi `public_test`.

**Bước 4 — Kiểm tra khả năng tái lập (`Kernel -> Restart & Run All`)**
Chạy kiểm thử tái lập cho từng notebook (hoặc kiểm tra lệnh `jupyter nbconvert --to notebook --execute`) và ghi lại kết quả pass/fail.

**Bước 5 — Lập khung Kết luận & Hành động đề xuất và lưu `storytelling_final_checklist.csv`**
Tóm tắt các kết luận rút ra từ bằng chứng 3 hồi, xuất bảng nghiệm thu cuối cùng vào `PHASE4_TABLES_DIR / 'storytelling_final_checklist.csv'`.

### Gợi ý phương pháp & Lưu ý
- Mọi khẳng định trong mục *Kết luận / Hành động* phải gắn kèm bằng chứng cụ thể từ Hồi 1, Hồi 2 hoặc Hồi 3 — không viết nhận định chung chung không có số liệu.
- Kiểm tra lại lần cuối xem có còn cell nào chưa điền `TODO`, bị lỗi font tiếng Việt hoặc chứa đường dẫn tuyệt đối trên máy cá nhân không.

### Triển khai


In [ ]:
# T4.1c — Bước 1: rà soát mạch kể chuyện
story_map = pd.DataFrame([
    ["Hồi 1", "Dataset discovery", "Bối cảnh: 14 doanh nghiệp public_test và phạm vi dữ liệu"],
    ["Hồi 2", "Business EDA", "Vấn đề/đặc trưng ở cấp doanh nghiệp đại diện"],
    ["Hồi 3", "Cross-business", "Bằng chứng đối chiếu về weekend, nhịp vận hành và Tết"],
    ["Hồi 4", "Storytelling", "Kiểm định biểu đồ, xuất bản và tổng hợp kết luận/hành động"],
], columns=["act","phase","story_role"])
display(story_map)


In [ ]:
# T4.1c — Bước 2: kiểm tra các chỉ số chốt từ Phase 3
ops_path = PHASE3_SUMMARY_DIR / "tables" / "operating_rhythm_summary.csv"
tet_path = PHASE3_SUMMARY_DIR / "tables" / "tet_window_comparison.csv"
limitations_path = PHASE3_SUMMARY_DIR / "tables" / "cross_business_limitations.csv"

ops_df = pd.read_csv(ops_path)
tet_df = pd.read_csv(tet_path)
limitations_df = pd.read_csv(limitations_path)

headline_metrics = {
    "operating_rhythm_rows": len(ops_df),
    "tet_window_rows": len(tet_df),
    "weekend_lift_min_pct": float(ops_df["weekend_vs_weekday_lift_pct"].min()),
    "weekend_lift_max_pct": float(ops_df["weekend_vs_weekday_lift_pct"].max()),
    "hourly_eligible_businesses": 2,
}
headline_metrics


In [ ]:
# T4.1c — Bước 3: giới hạn và thuật ngữ
terminology_limitations_df = pd.DataFrame([
    ["public_test", "Phạm vi dữ liệu công khai được phân tích"],
    ["cửa sổ chung 328 ngày", "Cửa sổ so sánh được dùng cho operating rhythm"],
    ["weekend vs weekday lift", "Mức tăng % trung bình đơn/ngày cuối tuần so với ngày thường"],
    ["HH:MM", "Chỉ dùng cho doanh nghiệp có giờ thực tế trong dữ liệu gốc"],
    ["không suy diễn nhân quả", "Kết quả chỉ mô tả quan sát, không chứng minh nguyên nhân"],
], columns=["term","usage"])
display(terminology_limitations_df)


In [ ]:
# T4.1c — Bước 4: kiểm tra khả năng tái lập ở mức artifact
reproducibility_check = pd.DataFrame([
    ["NB01","CHƯA CHẠY TRONG PHIÊN NÀY","Cần Restart & Run All trước khi nộp"],
    ["NB02","CHƯA CHẠY TRONG PHIÊN NÀY","Cần Restart & Run All trước khi nộp"],
    ["NB03","CHƯA CHẠY TRONG PHIÊN NÀY","Cần Restart & Run All trước khi nộp"],
    ["NB04","CHƯA CHẠY TRONG PHIÊN NÀY","Cần Restart & Run All trước khi nộp"],
], columns=["notebook","status","action"])
display(reproducibility_check)


In [ ]:
# T4.1c — Bước 5: storytelling_final_checklist.csv
final_checklist = pd.DataFrame([
    ["C1","Mạch kể chuyện","ĐẠT","Hồi 1 → Hồi 2 → Hồi 3 → Hồi 4"],
    ["C2","Phạm vi","ĐẠT","14 doanh nghiệp public_test; 6 DN đại diện cho đối chiếu"],
    ["C3","Cửa sổ so sánh","ĐẠT","328 ngày cho operating rhythm"],
    ["C4","Weekend vs weekday","ĐẠT","Lift quan sát từ +5.19% đến +55.57% trong 6 DN"],
    ["C5","Nhịp theo giờ","GIỚI HẠN","Chỉ 2/6 DN đại diện có HH:MM thực tế"],
    ["C6","Tết 2026","ĐẠT","Có bảng trước/trong/sau Tết cho 6 DN đại diện"],
    ["C7","Tết 2025","GIỚI HẠN","Chỉ dev_002 và dev_009 đủ cửa sổ theo artifact Phase 3"],
    ["C8","So sánh quy mô tuyệt đối","GIỚI HẠN","Không dùng tổng tuyệt đối; dùng %, Index, TB/ngày"],
    ["C9","Restart & Run All","CHƯA XÁC NHẬN","Chưa thực thi toàn bộ 4 notebook trong phiên này"],
    ["C10","Review thị giác","CHƯA XÁC NHẬN","Cần chốt các cột CẦN KIỂM TRA trong chart audit"],
], columns=["check_id","area","status","evidence"])
final_checklist.to_csv(PHASE4_TABLES_DIR / "storytelling_final_checklist.csv", index=False, encoding="utf-8-sig")
display(final_checklist)


### Kết quả & Nhận định

**Câu chuyện dữ liệu:** Hồi 1 xác định phạm vi 14 doanh nghiệp; Hồi 2 phân tích các doanh nghiệp đại diện; Hồi 3 cho thấy khác biệt về nhịp cuối tuần, khả năng phân tích theo giờ và biến động Tết; Hồi 4 kiểm định cách truyền đạt các bằng chứng đó.

**Điểm nổi bật từ Phase 3:**
- Weekend lift khác nhau đáng kể giữa các doanh nghiệp; F&B có mức tăng cao hơn trong các doanh nghiệp được so sánh.
- Phân tích theo giờ chỉ hợp lệ cho 2/6 doanh nghiệp đại diện vì các doanh nghiệp còn lại không có HH:MM thực tế trong dữ liệu gốc.
- Tết 2026 cho thấy phản ứng khác nhau theo doanh nghiệp; không nên khái quát thành một quy luật chung cho toàn bộ ngành.
- Không sử dụng tổng doanh thu/tổng đơn tuyệt đối để xếp hạng giữa các doanh nghiệp do khác quy mô, cấp ghi nhận và ngày cutoff.

**Tái lập:** Chưa xác nhận `Restart & Run All` cho cả 4 notebook trong phiên này; cần thực hiện trước khi nộp.


## 05. Tổng kết giai đoạn & Bàn giao

### Đã thực hiện
- [x] `T4.1a`: Quét danh mục biểu đồ Hồi 1–3 và tạo `chart_audit_checklist.csv`.
- [x] `T4.1b`: Xuất các ứng viên không phải draft sang `figures_300dpi/`, metadata DPI mục tiêu 300 và kiểm tra chữ ký PNG.
- [x] `T4.1c`: Tạo `storytelling_final_checklist.csv` và tổng hợp các giới hạn/điểm kiểm tra.
- [ ] Review thủ công toàn bộ tiêu chí thị giác.
- [ ] Restart & Run All cả 4 notebook.

### Thông điệp cốt lõi
Kết quả Phase 3 cho thấy hành vi vận hành khác nhau giữa các doanh nghiệp. Vì phạm vi và khả năng quan sát không đồng nhất, kết luận nên giữ ở mức mô tả trong `public_test`, không suy rộng toàn ngành và không diễn giải nhân quả.

### Giới hạn
- Chỉ phản ánh 14 doanh nghiệp trong `public_test` giai đoạn 2025–2026.
- Phân tích theo giờ chỉ có 2/6 DN đại diện đủ dữ liệu HH:MM.
- Không so sánh trực tiếp tổng tuyệt đối giữa các DN khác quy mô/cấp ghi nhận/cutoff.
- Tái lập toàn bộ 4 notebook chưa được xác nhận trong phiên này.
